
---
## Section 8 - Model Interpretability with SHAP

SHAP (SHapley Additive exPlanations) grounds each feature's contribution in
cooperative game theory. Unlike the aggregate MDI importances in Section 6,
SHAP values show **direction** (positive = pushes toward Team A winning) and
**magnitude** per prediction, giving an instance-level explanation that
is more faithful to how the model actually made each decision.

### Why three separate explainers?
| Model | Explainer | Reason |
|-------|-----------|--------|
| Logistic Regression | `LinearExplainer` | Exploits linear structure; exact computation |
| Random Forest | `TreeExplainer` | Tree-path algorithm; uses **unscaled** features |
| XGBoost | `TreeExplainer` | Same tree-path algorithm; uses **unscaled** features |

> **Interpretation note:** SHAP values are in *log-odds space* for LR and in
> *probability space* for tree models. Mean |SHAP| across all 501 test matches
> gives a global feature importance ranking that is comparable across models.

In [ ]:
# 8.1 SHAP Value Computation
# Install shap if not present
try:
    import shap
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'shap'])
    import shap

import warnings
warnings.filterwarnings('ignore')

# ── Logistic Regression (scaled features) ───────────────────────────────
lr_explainer = shap.LinearExplainer(lr_model, X_train_sc,
                                    feature_perturbation='interventional')
lr_shap_vals = lr_explainer.shap_values(X_test_sc)          # (n_test, 13)

# ── Random Forest (unscaled features) ───────────────────────────────────
rf_explainer = shap.TreeExplainer(rf_model, feature_perturbation='tree_path_dependent')
rf_shap_raw  = rf_explainer.shap_values(X_test, check_additivity=False)

# Handle both old-API (list of arrays) and new-API (3-D array)
if isinstance(rf_shap_raw, list):
    rf_shap_vals = rf_shap_raw[1]                           # class-1 slice
elif hasattr(rf_shap_raw, 'ndim') and rf_shap_raw.ndim == 3:
    rf_shap_vals = rf_shap_raw[:, :, 1]
else:
    rf_shap_vals = rf_shap_raw

# ── XGBoost (unscaled features) ─────────────────────────────────────────
xgb_explainer = shap.TreeExplainer(xgb_model)
xgb_shap_vals = xgb_explainer.shap_values(X_test)          # (n_test, 13)

# Convert any DataFrames to numpy for consistent indexing
X_test_np = X_test.values if hasattr(X_test, 'values') else X_test

# Mean |SHAP| summary table
shap_summary = pd.DataFrame({
    'Feature'        : FEATURES,
    'LR  mean|SHAP|' : np.abs(lr_shap_vals).mean(axis=0),
    'RF  mean|SHAP|' : np.abs(rf_shap_vals).mean(axis=0),
    'XGB mean|SHAP|' : np.abs(xgb_shap_vals).mean(axis=0),
})
shap_summary['Avg mean|SHAP|'] = shap_summary[
    ['LR  mean|SHAP|', 'RF  mean|SHAP|', 'XGB mean|SHAP|']
].mean(axis=1)
shap_summary = shap_summary.sort_values('Avg mean|SHAP|', ascending=False).reset_index(drop=True)

print('SHAP values computed successfully.')
print(f'  LR  shap shape : {lr_shap_vals.shape}')
print(f'  RF  shap shape : {rf_shap_vals.shape}')
print(f'  XGB shap shape : {xgb_shap_vals.shape}')
print()
print('Mean |SHAP| ranking (averaged across models):')
print(shap_summary[['Feature', 'LR  mean|SHAP|', 'RF  mean|SHAP|',
                     'XGB mean|SHAP|', 'Avg mean|SHAP|']].to_string(index=False))

In [ ]:
# 8.2 SHAP Beeswarm Plots - per-match feature contributions (all three models)
# Colour: red = high feature value, blue = low feature value (SHAP convention)

DARK_BG  = '#0d1117'
PANEL_BG = '#161b22'
EDGE_CLR = '#30363d'
TXT_CLR  = '#e6edf3'

plt.rcParams.update({
    'figure.facecolor': DARK_BG,  'axes.facecolor'  : PANEL_BG,
    'axes.edgecolor'  : EDGE_CLR, 'text.color'      : TXT_CLR,
    'axes.labelcolor' : TXT_CLR,  'xtick.color'     : TXT_CLR,
    'ytick.color'     : TXT_CLR,  'grid.color'      : EDGE_CLR,
    'axes.titlecolor' : TXT_CLR,
})

MODEL_DATA = [
    ('Logistic Regression', lr_shap_vals,  X_test_sc,       FEATURES),
    ('Random Forest',       rf_shap_vals,  X_test_np,       FEATURES),
    ('XGBoost',             xgb_shap_vals, X_test_np,       FEATURES),
]

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle(
    'SHAP Beeswarm - Feature Contributions per Match (2025 Test Set)',
    fontsize=14, color=TXT_CLR, fontweight='bold', y=1.01
)

np.random.seed(42)
for ax, (model_name, sv, data_arr, feat_names) in zip(axes, MODEL_DATA):
    # Sort by mean |SHAP| ascending so highest-impact feature is at the top
    order       = np.argsort(np.abs(sv).mean(axis=0))
    sv_sorted   = sv[:, order]
    data_sorted = data_arr[:, order]
    feat_sorted = [feat_names[i] for i in order]
    n_feat      = len(feat_sorted)

    for fi in range(n_feat):
        shap_col = sv_sorted[:, fi]
        feat_col = data_sorted[:, fi]
        feat_min, feat_max = feat_col.min(), feat_col.max()
        norm   = (feat_col - feat_min) / (feat_max - feat_min + 1e-9)
        colors = plt.cm.RdYlBu_r(norm)                    # red=high, blue=low
        jitter = np.random.uniform(-0.3, 0.3, size=len(shap_col))
        ax.scatter(shap_col, fi + jitter,
                   c=colors, s=6, alpha=0.6, linewidths=0, rasterized=True)

    ax.set_yticks(range(n_feat))
    ax.set_yticklabels(feat_sorted, fontsize=8.5)
    ax.axvline(0, color=EDGE_CLR, linewidth=0.8, linestyle='--')
    ax.set_xlabel('SHAP value  (+ → Team A win)', fontsize=8.5)
    ax.set_title(model_name, fontsize=11, fontweight='bold', pad=8)
    ax.set_facecolor(PANEL_BG)
    for spine in ax.spines.values():
        spine.set_edgecolor(EDGE_CLR)

# Shared feature-value colour bar
sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, orientation='horizontal',
                    fraction=0.02, pad=0.12, aspect=50)
cbar.set_label('Feature value  (red = high, blue = low)', color=TXT_CLR, fontsize=9)
cbar.ax.xaxis.set_tick_params(color=TXT_CLR)
plt.setp(cbar.ax.get_xticklabels(), color=TXT_CLR, fontsize=8)
cbar.outline.set_edgecolor(EDGE_CLR)

plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight',
            facecolor=DARK_BG, edgecolor='none')
plt.show()
print('Saved: shap_beeswarm.png')

In [ ]:
# 8.3 Mean |SHAP| Global Feature Importance - all models side-by-side

ACCENT_COLORS = ['#58a6ff', '#3fb950', '#f0883e']   # LR=blue, RF=green, XGB=orange
BAR_MODELS    = ['LR  mean|SHAP|', 'RF  mean|SHAP|', 'XGB mean|SHAP|']
MODEL_LABELS  = ['Logistic Regression', 'Random Forest', 'XGBoost']

plot_df = shap_summary.sort_values('Avg mean|SHAP|', ascending=True)

fig, (ax_bar, ax_table) = plt.subplots(
    1, 2, figsize=(18, 6),
    gridspec_kw={'width_ratios': [2, 1]}
)
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('Mean |SHAP| Feature Importance - All Models (2025 Test Set)',
             fontsize=13, color=TXT_CLR, fontweight='bold')

n_feat = len(plot_df)
y_pos  = np.arange(n_feat)
bar_h  = 0.25

for i, (col, label, color) in enumerate(zip(BAR_MODELS, MODEL_LABELS, ACCENT_COLORS)):
    offsets = y_pos + (i - 1) * bar_h
    ax_bar.barh(offsets, plot_df[col], height=bar_h * 0.9,
                label=label, color=color, alpha=0.85)

ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(plot_df['Feature'], fontsize=9)
ax_bar.set_xlabel('Mean |SHAP| value', fontsize=10)
ax_bar.set_facecolor(PANEL_BG)
ax_bar.legend(loc='lower right', fontsize=9,
              facecolor=PANEL_BG, edgecolor=EDGE_CLR, labelcolor=TXT_CLR)
ax_bar.tick_params(colors=TXT_CLR)
for spine in ax_bar.spines.values():
    spine.set_edgecolor(EDGE_CLR)

# Rank table on the right
ax_table.set_facecolor(PANEL_BG)
ax_table.axis('off')

top_disp = shap_summary[['Feature', 'LR  mean|SHAP|', 'RF  mean|SHAP|', 'XGB mean|SHAP|']].copy()
top_disp.insert(0, 'Rank', range(1, len(top_disp) + 1))
top_disp.columns = ['Rank', 'Feature', 'LR', 'RF', 'XGB']
for col in ['LR', 'RF', 'XGB']:
    top_disp[col] = top_disp[col].apply(lambda x: f'{x:.4f}')

tbl = ax_table.table(
    cellText  = top_disp.values,
    colLabels = top_disp.columns,
    cellLoc   = 'center',
    loc       = 'center',
    bbox      = [0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor(EDGE_CLR)
    if row == 0:
        cell.set_facecolor('#21262d')
        cell.set_text_props(color=TXT_CLR, fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#1c2128')
        cell.set_text_props(color=TXT_CLR)
    else:
        cell.set_facecolor(PANEL_BG)
        cell.set_text_props(color=TXT_CLR)

plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight',
            facecolor=DARK_BG, edgecolor='none')
plt.show()
print('Saved: shap_bar.png')
print()
print('Top 5 features by average mean|SHAP|:')
print(shap_summary[['Feature', 'Avg mean|SHAP|']].head(5).to_string(index=False))


---
### 8.4 SHAP Findings - Top Feature Interpretation

The mean |SHAP| rankings confirm and extend the MDI analysis from Section 6.

| Rank | Feature | Interpretation |
|------|---------|---------------|
| 1 | `hist_win_rate_diff` | **Strongest predictor across all three models.** A positive value (Team A has a higher historical win rate) consistently pushes the prediction toward a Team A win. The beeswarm plots show a near-linear relationship: the wider the spread of SHAP values for this feature, the more it dominates individual predictions. |
| 2 | `hist_rating_diff` | Player-level skill differential. High feature value (red in beeswarm) = Team A's roster has historically outperformed Team B's - large positive SHAP contribution. Aligns with *Finding 2* in Section 8: relative strength dominates absolute performance. |
| 3 | `map_win_pct_diff` | Captures map-level consistency. Highly correlated with `hist_win_rate_diff` (r ≈ 0.80 from EDA), which is why LR assigns it lower relative weight - regularisation suppresses redundant features. Tree models still extract independent signal from it via split interactions. |
| 4 | `ta_hist_win_rate` / `tb_hist_win_rate` | The individual win-rate components add marginal signal beyond the differential, especially for LR which treats each feature independently. |
| 5–13 | Context features (`stage_stakes`, `is_elimination_match`, `is_grand_final`, `ta_ban_first`) | Near-zero mean |SHAP| across all models, confirming *Finding 3*: match-context features contribute very little predictive lift beyond team-quality differentials. |

**Model agreement:** All three models rank `hist_win_rate_diff` and `hist_rating_diff` as the top two features, providing strong cross-model validation. The small divergence between LR (which weights absolute features more) and the tree models (which focus on differentials via interaction splits) reflects the difference between linear vs. non-linear decision boundaries - as analysed in Section 4 and evidenced in the ROC-AUC comparison.

**What SHAP adds over MDI:** The beeswarm plots reveal that `hist_win_rate_diff` has a **monotone** SHAP profile - the higher the differential, the larger the positive SHAP value, with no reversals. This linearity explains why Logistic Regression achieves competitive AUC despite its simplicity: the most important feature is already near-linear in its effect on the log-odds of Team A winning.